In [ ]:
from langchain_anthropic import ChatAnthropicfrom langchain.agents import AgentExecutor, create_tool_calling_agentfrom langchain_core.prompts import ChatPromptTemplatefrom langchain.tools import tool@tooldef qof_lookup(condition: str) -> str:    mock_qof = {        "obesity": [            {"snomed_code": "414916001", "description": "Obesity (BMI 30+)",             "indicator_id": "OB001", "source": "QOF v49 2024-25"},            {"snomed_code": "238136002", "description": "Morbid obesity",             "indicator_id": "OB001", "source": "QOF v49 2024-25"},        ],        "type 2 diabetes": [            {"snomed_code": "44054006", "description": "Type 2 diabetes mellitus",             "indicator_id": "DM012", "source": "QOF v49 2024-25"},        ]    }    results = mock_qof.get(condition.lower(), [])    if not results:        return f"No QOF codes found for '{condition}'. Try a different spelling."    import json    return json.dumps(results, indent=2)@tooldef usage_lookup(snomed_code: str) -> str:    mock_usage = {        "414916001": {"annual_count": 3_200_000, "percentile": 98},        "238136002": {"annual_count": 450_000,   "percentile": 87},        "44054006":  {"annual_count": 4_823_109, "percentile": 99},    }    result = mock_usage.get(snomed_code, {"annual_count": 0, "percentile": 0})    if result["percentile"] >= 90:        result["verdict"] = "HIGH usage ? mainstream code"    elif result["percentile"] >= 50:        result["verdict"] = "MEDIUM usage ? used but not dominant"    else:        result["verdict"] = "LOW usage ? check if deprecated or too specific"    import json    return json.dumps(result, indent=2)SYSTEM_PROMPT = """You are a clinical informatics assistant for NICE (National Institute for Health and Care Excellence).Your task is to build a defensible clinical code list for a given research question.MANDATORY RULES ? follow these exactly:1. Always call qof_lookup FIRST for every condition mentioned.2. For every code you find, call usage_lookup to check how commonly it is recorded in real patient data.3. You MUST NOT recommend any code that was not returned by one of your tools.4. Assign a confidence tier to every code:   - HIGH: found in QOF AND has usage percentile >= 90   - MEDIUM: found in QOF OR has usage percentile >= 50   - REVIEW: everything else ? flag it for human checking"""llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT), ("human", "{input}"), ("placeholder", "{agent_scratchpad}")])tools = [qof_lookup, usage_lookup]agent = create_tool_calling_agent(llm, tools, prompt)agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=10, return_intermediate_steps=True)result = agent_executor.invoke({"input": "Build a code list for patients with obesity who also have type 2 diabetes."})print("\n" + "="*60)print("FINAL OUTPUT:")print("="*60)print(result["output"])